# M12: HAR-RV-J — Jump-Augmented Volatility Forecasting on QC Cloud

**Model**: HAR-RV-J (Andersen, Bollerslev & Diebold 2007, "Roughing It Up")
**Local verdict**: BEATS (p=7.9e-7, 64/84 wins, median delta-Sharpe +0.0032 vs HAR Classic)
**Goal**: Reproduce M12 BEATS verdict on QC Cloud data, compare with local results.

## Architecture

```
RV_t = BPV_t + J_t  (Huang-Tauchen bipower decomposition)
J_t = max(RV_t - mu*BPV_t, 0),  mu = pi/2 * (1 + 1/n) ≈ 0.6

log(RV_{t+h}) = b0 + b_d*log(RV_t) + b_w*log(RV_w) + b_m*log(RV_m)
               + b_dj*J_t + b_wj*J_w + b_mj*J_m + e
```

- Walk-forward 5-fold expanding OLS
- 7 coins (BTC, ETH, SOL, LTC, XRP, ADA, DOT) × 3 horizons (h=1,5,10)
- Kelly cap=1.0, fee=50bps, sign-test vs HAR Classic

In [ ]:
# Section 1: Setup — QC Cloud QuantBook
from AlgorithmImports import *
from QuantConnect.Research import *
import numpy as np
import pandas as pd
from scipy import stats

qb = QuantBook()
print(f"QC Cloud ready — User: {qb.UserId}")

In [ ]:
# Section 2: Load crypto hourly data via QuantBook
# BTC-USD as primary (Bitstamp/Coinbase)
coins = ["BTCUSD", "ETHUSD", "SOLUSD", "LTCUSD", "XRPUSD", "ADAUSD", "DOTUSD"]
crypto_data = {}

for symbol_str in coins:
    try:
        crypto = qb.AddCrypto(symbol_str, Resolution.Hour, Market.Bitstamp)
        history = qb.History(crypto.Symbol, timedelta(days=3650), Resolution.Hour)
        if len(history) > 0:
            crypto_data[symbol_str] = history
            print(f"{symbol_str}: {len(history)} hourly bars loaded")
        else:
            # Fallback to Coinbase
            crypto = qb.AddCrypto(symbol_str, Resolution.Hour, Market.Coinbase)
            history = qb.History(crypto.Symbol, timedelta(days=3650), Resolution.Hour)
            crypto_data[symbol_str] = history
            print(f"{symbol_str}: {len(history)} hourly bars (Coinbase)")
    except Exception as e:
        print(f"{symbol_str}: unavailable — {e}")

print(f"\nTotal coins loaded: {len(crypto_data)}")

In [ ]:
# Section 3: Realized Variance & Bipower Variation reconstruction
def compute_daily_rv_bpv(hourly_df):
    """Compute daily RV and BPV from hourly log returns."""
    df = hourly_df.copy()
    df["log_ret"] = np.log(df["close"] / df["close"].shift(1))
    df = df.dropna(subset=["log_ret"])
    
    # Group by date
    df["date"] = df.index.date if hasattr(df.index, "date") else pd.to_datetime(df.index).date
    
    daily = df.groupby("date").agg(
        rv=("log_ret", lambda x: np.sum(x**2)),
        n_bars=("log_ret", "count"),
        close=("close", "last"),
    )
    
    # BPV: pi/2 * sum(|r_t| * |r_{t-1}|) (Huang-Tauchen)
    mu_bpv = np.pi / 2  # asymptotic multiplier
    bpv_parts = []
    for date, group in df.groupby("date"):
        r = group["log_ret"].values
        if len(r) >= 2:
            bpv = mu_bpv * np.sum(np.abs(r[1:]) * np.abs(r[:-1]))
        else:
            bpv = np.nan
        bpv_parts.append({"date": date, "bpv": bpv})
    
    bpv_df = pd.DataFrame(bpv_parts).set_index("date")
    daily = daily.join(bpv_df)
    
    # Jump component: J_t = max(RV_t - BPV_t, 0)
    daily["jumps"] = np.maximum(daily["rv"] - daily["bpv"], 0)
    
    return daily

# Process all coins
daily_data = {}
for symbol, hist in crypto_data.items():
    daily = compute_daily_rv_bpv(hist)
    daily_data[symbol] = daily
    print(f"{symbol}: {len(daily)} trading days, "
          f"RV range [{daily['rv'].min():.6f}, {daily['rv'].max():.6f}]")

In [ ]:
# Section 4: HAR-RV-J model (walk-forward expanding OLS)
from sklearn.linear_model import LinearRegression

def har_rv_j_features(daily_df, horizon=1):
    """Build HAR-RV-J feature matrix: log(RV) lags + jump lags."""
    df = daily_df.copy()
    df["log_rv"] = np.log(df["rv"].clip(lower=1e-10))
    
    # HAR lag features (daily=1, weekly=5, monthly=22)
    df["rv_d"] = df["log_rv"]
    df["rv_w"] = df["log_rv"].rolling(5).mean()
    df["rv_m"] = df["log_rv"].rolling(22).mean()
    
    # Jump lag features
    df["j_d"] = df["jumps"]
    df["j_w"] = df["jumps"].rolling(5).mean()
    df["j_m"] = df["jumps"].rolling(22).mean()
    
    # Target: log(RV_{t+h})
    df["target"] = df["log_rv"].shift(-horizon)
    
    return df.dropna()

def walk_forward_har_rv_j(df, n_folds=5, horizon=1):
    """Walk-forward expanding window with n_folds."""
    features = ["rv_d", "rv_w", "rv_m", "j_d", "j_w", "j_m"]
    prepared = har_rv_j_features(df, horizon=horizon)
    
    n = len(prepared)
    fold_size = n // (n_folds + 1)
    
    predictions = []
    actuals = []
    
    for fold in range(1, n_folds + 1):
        split = fold_size * fold
        train = prepared.iloc[:split]
        test_start = split
        test_end = min(split + fold_size, n)
        test = prepared.iloc[test_start:test_end]
        
        if len(test) == 0:
            continue
        
        X_train = train[features].values
        y_train = train["target"].values
        X_test = test[features].values
        y_test = test["target"].values
        
        model = LinearRegression().fit(X_train, y_train)
        pred = model.predict(X_test)
        
        predictions.extend(pred)
        actuals.extend(y_test)
    
    return np.array(predictions), np.array(actuals)

print("HAR-RV-J model functions defined")

In [ ]:
# Section 5: HAR Classic baseline (3 regressors: log(RV_d), log(RV_w), log(RV_m))
def har_classic_features(daily_df, horizon=1):
    """Build HAR Classic feature matrix: log(RV) lags only."""
    df = daily_df.copy()
    df["log_rv"] = np.log(df["rv"].clip(lower=1e-10))
    
    df["rv_d"] = df["log_rv"]
    df["rv_w"] = df["log_rv"].rolling(5).mean()
    df["rv_m"] = df["log_rv"].rolling(22).mean()
    
    df["target"] = df["log_rv"].shift(-horizon)
    
    return df.dropna()

def walk_forward_har_classic(df, n_folds=5, horizon=1):
    """Walk-forward expanding window HAR Classic."""
    features = ["rv_d", "rv_w", "rv_m"]
    prepared = har_classic_features(df, horizon=horizon)
    
    n = len(prepared)
    fold_size = n // (n_folds + 1)
    
    predictions = []
    actuals = []
    
    for fold in range(1, n_folds + 1):
        split = fold_size * fold
        train = prepared.iloc[:split]
        test_start = split
        test_end = min(split + fold_size, n)
        test = prepared.iloc[test_start:test_end]
        
        if len(test) == 0:
            continue
        
        X_train = train[features].values
        y_train = train["target"].values
        X_test = test[features].values
        y_test = test["target"].values
        
        model = LinearRegression().fit(X_train, y_train)
        pred = model.predict(X_test)
        
        predictions.extend(pred)
        actuals.extend(y_test)
    
    return np.array(predictions), np.array(actuals)

print("HAR Classic baseline functions defined")

In [ ]:
# Section 6: Run M12 vs HAR Classic sweep
horizons = [1, 5, 10]
results = []

for symbol, daily in daily_data.items():
    for h in horizons:
        try:
            pred_jrv, actual_jrv = walk_forward_har_rv_j(daily, n_folds=5, horizon=h)
            pred_har, actual_har = walk_forward_har_classic(daily, n_folds=5, horizon=h)
            
            mse_jrv = np.mean((pred_jrv - actual_jrv)**2)
            mse_har = np.mean((pred_har - actual_har)**2)
            
            # Kelly-based Sharpe comparison
            rv_pred_jrv = np.exp(pred_jrv)  # back to RV level
            rv_pred_har = np.exp(pred_har)
            rv_actual = np.exp(actual_jrv)
            
            results.append({
                "coin": symbol,
                "horizon": h,
                "mse_har_rv_j": mse_jrv,
                "mse_har": mse_har,
                "mse_change_pct": (mse_jrv - mse_har) / mse_har * 100,
                "n_oos": len(actual_jrv),
            })
            
            print(f"{symbol} h={h}: MSE_HAR={mse_har:.6f}, MSE_JRV={mse_jrv:.6f}, "
                  f"change={results[-1]['mse_change_pct']:+.1f}%")
        except Exception as e:
            print(f"{symbol} h={h}: ERROR — {e}")

results_df = pd.DataFrame(results)
print(f"\nTotal combos: {len(results_df)}")
print(results_df.to_string())

In [ ]:
# Section 7: Kelly position sizing & Sharpe comparison
def kelly_sharpe(pred_rv, actual_rv, close_prices, fee_bps=50, cap=1.0, mu_window=60):
    """Kelly position sizing based on vol forecast, compute OOS Sharpe."""
    n = len(pred_rv)
    if n < mu_window + 10:
        return np.nan, np.nan
    
    # Signal: forecast vol vs rolling mean vol
    rolling_mu = pd.Series(actual_rv).rolling(mu_window).mean().values
    signal = np.where(pred_rv < rolling_mu, 1.0, -1.0)  # long if vol predicted low
    
    # Position sizing: inverse vol
    vol_forecast = np.sqrt(pred_rv) * np.sqrt(252)  # annualized
    vol_forecast = np.maximum(vol_forecast, 0.01)  # floor
    kelly_weight = np.minimum(1.0 / vol_forecast, cap)
    
    # Returns
    returns = np.diff(np.log(close_prices[-(n+1):]))[:n]
    if len(returns) < n:
        returns = np.pad(returns, (0, n - len(returns)), constant_values=0)
    
    # Apply Kelly sizing and signal
    weighted_returns = signal * kelly_weight * returns
    
    # Net of fees
    fee = fee_bps / 10000
    trades = np.abs(np.diff(np.concatenate([[0], signal * kelly_weight])))
    fee_costs = np.concatenate([[0], trades * fee])[:n]
    net_returns = weighted_returns - fee_costs
    
    sharpe = np.mean(net_returns) / (np.std(net_returns) + 1e-10) * np.sqrt(252)
    return sharpe, np.mean(net_returns)

# Compute Sharpe for each combo
for i, row in results_df.iterrows():
    symbol = row["coin"]
    h = row["horizon"]
    daily = daily_data[symbol]
    close = daily["close"].values
    
    pred_jrv, actual_jrv = walk_forward_har_rv_j(daily, n_folds=5, horizon=h)
    pred_har, actual_har = walk_forward_har_classic(daily, n_folds=5, horizon=h)
    
    rv_pred_jrv = np.exp(pred_jrv)
    rv_pred_har = np.exp(pred_har)
    rv_actual = np.exp(actual_jrv)
    
    # Align close prices
    prepared = har_rv_j_features(daily, horizon=h)
    n_oos = len(pred_jrv)
    close_oos = close[-(n_oos+1):]
    
    sharpe_jrv, _ = kelly_sharpe(rv_pred_jrv, rv_actual, close_oos, fee_bps=50)
    sharpe_har, _ = kelly_sharpe(rv_pred_har, rv_actual, close_oos, fee_bps=50)
    
    results_df.loc[i, "sharpe_har_rv_j"] = sharpe_jrv
    results_df.loc[i, "sharpe_har"] = sharpe_har
    results_df.loc[i, "delta_sharpe"] = sharpe_jrv - sharpe_har
    results_df.loc[i, "beats"] = int(sharpe_jrv > sharpe_har)

print("Kelly Sharpe comparison complete")
print(results_df[["coin", "horizon", "sharpe_har_rv_j", "sharpe_har", "delta_sharpe", "beats"]].to_string())

In [ ]:
# Section 8: Sign-test & verdict
from scipy.stats import binomtest

beats = results_df["beats"].sum()
total = len(results_df)
win_rate = beats / total if total > 0 else 0

if total > 0:
    p_value = binomtest(beats, total, 0.5).pvalue
else:
    p_value = 1.0

median_delta_sharpe = results_df["delta_sharpe"].median()

print("=" * 60)
print(f"M12 HAR-RV-J vs HAR Classic — QC Cloud Verdict")
print(f"={'=' * 60}")
print(f"Beats: {beats}/{total} ({win_rate:.1%})")
print(f"Sign-test p-value: {p_value:.4f}")
print(f"Median delta-Sharpe: {median_delta_sharpe:+.4f}")
print(f"\nBEATS criteria: p < 0.05 AND win_rate >= 60%")

if p_value < 0.05 and win_rate >= 0.60:
    print(f"VERDICT: BEATS (p={p_value:.4f}, win={win_rate:.1%})")
else:
    print(f"VERDICT: NO BEATS (p={p_value:.4f}, win={win_rate:.1%})")

# Per-horizon breakdown
print(f"\nPer-horizon breakdown:")
for h in horizons:
    subset = results_df[results_df["horizon"] == h]
    if len(subset) > 0:
        h_beats = subset["beats"].sum()
        h_total = len(subset)
        h_win = h_beats / h_total
        h_med_ds = subset["delta_sharpe"].median()
        print(f"  h={h}: {h_beats}/{h_total} ({h_win:.1%}), "
              f"median delta-Sharpe={h_med_ds:+.4f}")

# Comparison with local result
print(f"\nLocal M12 result: BEATS p=7.9e-7, 64/84 (76.2%), delta-Sharpe +0.0032")
print(f"QC Cloud result:  {beats}/{total} ({win_rate:.1%}), p={p_value:.4f}, "
      f"delta-Sharpe={median_delta_sharpe:+.4f}")

## Interpretation

This notebook reproduces the M12 HAR-RV-J verdict on QC Cloud data.
Key comparison points with local Python results:

1. **Data source**: QC Cloud crypto hourly (Bitstamp/Coinbase) vs local Bitstamp tick-aggregated
2. **RV reconstruction**: QC Hour-resolution (24 bars/day) vs local tick-resolution. Expected mismatch < 5% for BTC.
3. **Jump detection**: Identical Huang-Tauchen bipower variation method
4. **Walk-forward**: Same 5-fold expanding OLS protocol

If QC Cloud verdict matches local BEATS (p<0.05, win>=60%), proceed to Phase B:
deploy as QC algorithm `HarRvJSingleAsset` for backtested trading.